# 💰 Fintra-AI: Financial Analysis & Behavioral Insights (02_financial_analysis.ipynb)

### 🎯 Objective
This notebook performs comprehensive personal finance analytics across:
1. **Transaction Metrics**: Total volume, average ticket sizes, and merchant spending rankings.
2. **Expense Analysis**: Monthly burn rates, category rankings, and recurring expense heuristics.
3. **Income Analysis**: Monthly cash inflows, income source diversity, and income stability index ($CV$).
4. **Financial Behavior**: Net savings rate, 50/30/20 rule allocation, budget utilization, and high-spend burst periods.


In [1]:
# 1. Setup and Imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))

from ml.analysis.data_loader import load_project_dataset, generate_sample_financial_dataset
from ml.analysis.transaction_analyzer import (
    get_transaction_summary,
    analyze_transaction_frequency,
    analyze_category_distribution,
    analyze_merchants
)
from ml.analysis.expense_analyzer import (
    aggregate_monthly_expenses,
    rank_category_spending,
    calculate_spending_trends,
    detect_recurring_expenses
)
from ml.analysis.income_analyzer import (
    aggregate_monthly_income,
    analyze_income_sources,
    calculate_income_stability,
    compare_income_vs_expenses
)
from ml.analysis.financial_behavior import (
    calculate_savings_rate,
    analyze_temporal_spending_patterns,
    analyze_50_30_20_compliance,
    evaluate_budget_adherence,
    identify_high_spending_periods
)

# Load dataset
df = load_project_dataset(include_raw_sources=True)
if df.empty or len(df[df['type'] == 'INCOME']) == 0:
    df = generate_sample_financial_dataset(n_records=400, seed=42)

print(f"Loaded {len(df)} transactions for financial analysis.")


Loaded 10440 transactions for financial analysis.


## 1. Core Transaction Summary & Merchant Analysis

We calculate global transaction volume, average amounts per transaction type, and top merchants by transaction frequency and monetary volume.


In [2]:
# Summary Metrics
summary = get_transaction_summary(df)
for k, v in summary.items():
    print(f"- {k.replace('_', ' ').title()}: {v:,.2f}" if isinstance(v, float) else f"- {k.replace('_', ' ').title()}: {v:,}")


- Total Transactions: 10,440
- Total Income Count: 125
- Total Expense Count: 10,315
- Total Volume Amount: 107,177,833.01
- Average Transaction Amount: 10,266.08
- Average Income Amount: 24,339.18
- Average Expense Amount: 10,095.53


In [3]:
# Top Spending Merchants
merchant_analysis = analyze_merchants(df, top_n=8)
print("--- Top Merchants by Spending (INR) ---")
print(merchant_analysis["by_spending"])

print("\n--- Top Merchants by Transaction Frequency ---")
print(merchant_analysis["by_frequency"])


--- Top Merchants by Spending (INR) ---
           merchant  transaction_count  total_amount  avg_amount
0     Samsung Store                 78    6206966.91    79576.50
1             Noise                 62    5768806.78    93045.27
2          HP World                 68    5631535.94    82816.70
3           OnePlus                 63    4948920.36    78554.29
4        Asus Store                 65    4809842.06    73997.57
5    Dell Exclusive                 65    4640370.72    71390.32
6  Reliance Digital                 57    4629056.62    81211.52
7             Croma                 59    4532158.58    76816.25

--- Top Merchants by Transaction Frequency ---
        merchant  transaction_count  total_amount  avg_amount
0           Milk                162       9047.00       55.85
1           auto                136       5720.00       42.06
2         snacks                115       5980.75       52.01
3          Other                114      87025.28      763.38
4        Grocery 

## 2. Expense Dynamics & Recurring Subscriptions

We break down monthly expenditure, identify top category contributors, analyze Month-over-Month (MoM) spending growth, and apply recurring payment heuristics.


In [4]:
# Monthly Expense & Category Ranking
monthly_exp = aggregate_monthly_expenses(df)
cat_ranking = rank_category_spending(df)

print("--- Monthly Expense Aggregation ---")
print(monthly_exp)

print("\n--- Category Ranking & Spending Share ---")
print(cat_ranking)


--- Monthly Expense Aggregation ---
      month  total_expense  transaction_count  average_expense
0   2015-01      135104.00                 73          1850.74
1   2015-02       78919.00                 54          1461.46
2   2015-03       39253.80                 42           934.61
3   2015-04       26806.00                 24          1116.92
4   2015-05       52603.00                 39          1348.79
5   2015-06        4318.00                  9           479.78
6   2015-07       22115.00                 17          1300.88
7   2015-08       20624.00                 15          1374.93
8   2015-09        5185.00                 18           288.06
9   2015-10       25592.80                 50           511.86
10  2015-11        4822.00                 20           241.10
11  2015-12        6645.00                 21           316.43
12  2016-01      264920.00                 33          8027.88
13  2016-02       22066.00                 22          1003.00
14  2016-03        

In [5]:
# Spending Trends & Growth Rates
trends = calculate_spending_trends(df)
print(trends[["month", "total_expense", "mom_growth_amount", "mom_growth_pct", "trend_direction"]])


      month  total_expense  mom_growth_amount  mom_growth_pct trend_direction
0   2015-01      135104.00               0.00            0.00          STABLE
1   2015-02       78919.00          -56185.00          -41.59      DECREASING
2   2015-03       39253.80          -39665.20          -50.26      DECREASING
3   2015-04       26806.00          -12447.80          -31.71      DECREASING
4   2015-05       52603.00           25797.00           96.24      INCREASING
5   2015-06        4318.00          -48285.00          -91.79      DECREASING
6   2015-07       22115.00           17797.00          412.16      INCREASING
7   2015-08       20624.00           -1491.00           -6.74      DECREASING
8   2015-09        5185.00          -15439.00          -74.86      DECREASING
9   2015-10       25592.80           20407.80          393.59      INCREASING
10  2015-11        4822.00          -20770.80          -81.16      DECREASING
11  2015-12        6645.00            1823.00           37.81   

In [6]:
# Recurring Expense Detection
recurring = detect_recurring_expenses(df, min_occurrences=2)
print(f"Identified {len(recurring)} recurring subscription/bill patterns:")
print(recurring)


Identified 10 recurring subscription/bill patterns:
               merchant       category  occurrences  mean_amount  amount_std  \
0      Small Cap fund 2          other           10      5000.00        0.00   
1      Small cap fund 1          other           10      5000.00        0.00   
2        Life Insurance          other            7     11077.71       85.03   
3  Equity Mutual Fund C          other            6      1000.00        0.00   
4      Kindle unlimited  entertainment            4       169.00        0.00   
5             Household          bills            3       364.67      306.49   
6              Cable TV  entertainment            3       261.00        0.00   
7      Self-development      education            2      1178.50      228.50   
8               Audible  entertainment            2       199.00        0.00   
9      garbage disposal          bills            2        33.50       16.50   

   interval_mean_days  interval_std_days    cadence confidence  
0 

## 3. Income Analysis & Stability Metrics

Income stability is quantified using the **Coefficient of Variation ($CV = \frac{\sigma}{\mu}$)** across monthly cash inflows.


In [7]:
# Income Analysis
stability = calculate_income_stability(df)
print("--- Income Stability Report ---")
for k, v in stability.items():
    print(f"  - {k}: {v}")

print("\n--- Income vs. Expense Comparison ---")
comparison = compare_income_vs_expenses(df)
print(comparison)


--- Income Stability Report ---
  - months_analyzed: 45
  - mean_monthly_income: 67608.83
  - std_monthly_income: 43124.6
  - coefficient_of_variation: 0.6379
  - min_monthly_income: 1127.0
  - max_monthly_income: 285385.0
  - stability_tier: HIGH_VOLATILITY
  - description: Irregular or variable freelance/commission-based income stream.

--- Income vs. Expense Comparison ---
      month  total_income  total_expense  net_savings  savings_rate_pct  \
0   2015-01          0.00      135104.00   -135104.00           -100.00   
1   2015-02      49806.00       78919.00    -29113.00            -58.45   
2   2015-03      70806.00       39253.80     31552.20             44.56   
3   2015-04      54106.00       26806.00     27300.00             50.46   
4   2015-05      47859.00       52603.00     -4744.00             -9.91   
5   2015-06      47859.00        4318.00     43541.00             90.98   
6   2015-07      47859.00       22115.00     25744.00             53.79   
7   2015-08      4980

## 4. Financial Behavior, Savings Rate & 50/30/20 Compliance

We compute:
- Net savings amount & savings rate percentage
- 50/30/20 budget framework compliance (Needs vs Wants vs Savings)
- Budget adherence & utilization
- High-spending anomaly burst periods


In [8]:
# Savings Rate & Health Assessment
total_inc = df[df["type"] == "INCOME"]["amount"].sum()
total_exp = df[df["type"] == "EXPENSE"]["amount"].sum()

savings_metrics = calculate_savings_rate(total_inc, total_exp)
for k, v in savings_metrics.items():
    print(f"- {k}: {v}")


- total_income: 3042397.35
- total_expenses: 104135435.66
- net_savings: -101093038.31
- savings_rate_pct: -3322.81
- health_status: DEFICIT


In [9]:
# 50/30/20 Allocation Compliance
compliance = analyze_50_30_20_compliance(df, lifestyle="balanced")
print(f"Lifestyle Profile: {compliance.get('lifestyle_profile', 'balanced').title()}")
print("Actual Allocation (%):", compliance.get("actual_allocation", {}))
print("Target Allocation (%):", compliance.get("target_allocation", {}))
print("Variance (%):          ", compliance.get("variance", {}))


Lifestyle Profile: Balanced
Actual Allocation (%): {'needs_amount': 27249296.65, 'wants_amount': 76886139.01, 'savings_amount': 0.0, 'needs_pct': 895.65, 'wants_pct': 2527.16, 'savings_pct': 0.0}
Target Allocation (%): {'needs_pct': 50.0, 'wants_pct': 30.0, 'savings_pct': 20.0}
Variance (%):           {'needs_variance_pct': 845.65, 'wants_variance_pct': 2497.16, 'savings_variance_pct': -20.0}


In [10]:
# Budget Adherence Matrix
budget_matrix = evaluate_budget_adherence(df)
print(budget_matrix)


        category  allocated_budget  actual_spend     variance  \
0       shopping       15620315.35   73318184.08 -57697868.73   
1      transport       15620315.35   13201836.14   2418479.21   
2     healthcare       15620315.35    8652310.63   6968004.72   
3           food       15620315.35    2781941.15  12838374.20   
4          other       15620315.35    2712674.13  12907641.22   
5          bills       15620315.35    2610314.73  13010000.62   
6  entertainment       15620315.35     585422.80  15034892.55   
7     investment       15620315.35     269858.00  15350457.35   
8      education       15620315.35       2894.00  15617421.35   

   utilization_pct         status  
0           469.38    OVER_BUDGET  
1            84.52  WITHIN_BUDGET  
2            55.39  WITHIN_BUDGET  
3            17.81  WITHIN_BUDGET  
4            17.37  WITHIN_BUDGET  
5            16.71  WITHIN_BUDGET  
6             3.75  WITHIN_BUDGET  
7             1.73  WITHIN_BUDGET  
8             0.02  WITHI

In [11]:
# High-Spending Burst Dates (IQR Thresholding)
spikes = identify_high_spending_periods(df, period="D")
print(f"Detected {len(spikes)} high-spending burst dates:")
print(spikes.head(10))


Detected 99 high-spending burst dates:
       period  total_spend  tx_count  upper_threshold  excess_spend
0  2025-03-31    943490.73        26        341586.46     601904.27
1  2025-06-05    920121.78        26        341586.46     578535.32
2  2025-09-02    815487.80        26        341586.46     473901.34
3  2025-07-14    740798.19        23        341586.46     399211.73
4  2025-08-07    701576.79        27        341586.46     359990.33
5  2025-01-23    650922.31        27        341586.46     309335.85
6  2025-09-09    647996.46        29        341586.46     306410.00
7  2025-11-08    640756.13        25        341586.46     299169.67
8  2025-10-27    620015.89        23        341586.46     278429.43
9  2025-09-15    603531.17        24        341586.46     261944.71
